In [1]:
import sys
sys.path.append('/home/users/zwb3/research/falling-models/')

import pandas as pd
import numpy as np
import itertools

from falling_trees import falling_rashomon_cpp as gravitree
from falling_trees.frl_rashomon_set_alg_falling_constraint_v2 import print_tree

In [2]:
def generate_XOR_truth_table(spine_length=4, n_xor=2):
    r"""Create a truth table for the DGP defined by an spine of given length
    with XOR interactions at the leaves.
    
    For example, with spine_length=3 and n_xor=2, we have the following DGP:
    
               x0
              /     \
            x1          x3
            / \        /      \
           x2  x2      x4          x6
           / \ / \    / \          / \
          0  1 1  0   x5 x5       x7   1
                    / \ / \      / \    
                   0  1 1  0    x8  x8
                               / \ / \
                              0  1 1  0
    
    The spine is x0, x3, x6.
    """

    features = list(range(spine_length * (n_xor + 1)))
    features = [f'X{i}' for i in features]
    spine_variables = [(2**(n_xor-1)+1)*i for i in range(spine_length)]
    
    print('Features:', features)
    print(f'Spine variables: {spine_variables}')
    
    def compute_y(row):
        for spine_var in spine_variables:
            if row[f'X{spine_var}'] == 1:
                xor_vars = [spine_var + i for i in range(1, n_xor + 1)]
                xor_result = 0
                for var in xor_vars:
                    xor_result ^= row[f'X{var}']
                return xor_result
        return 1
    
    X = pd.DataFrame(list(itertools.product([0, 1], repeat=len(features))), columns=features)
    y = X.apply(compute_y, axis=1)
    
    return X, y

X, y = generate_XOR_truth_table(spine_length=3, n_xor=2)
df = X.copy()
df['y'] = y
df

Features: ['X0', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']
Spine variables: [0, 3, 6]


,X0,X1,X2,X3,X4,X5,X6,X7,X8,y
0,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,0,0,1,1
2,0,0,0,0,0,0,0,1,0,1
3,0,0,0,0,0,0,0,1,1,1
4,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...
507,1,1,1,1,1,1,0,1,1,0
508,1,1,1,1,1,1,1,0,0,0
509,1,1,1,1,1,1,1,0,1,0
510,1,1,1,1,1,1,1,1,0,0


In [3]:
lam=0.0
eps=0.01
depth=10
branching_cost=1e-6
min_support=0.
rule_list_mode=False
use_heap=True
max_cache_size=10**7
dump_path='/dev/null'

result = gravitree.run_from_xy(
        X.values,
        y.values,
        lam=lam,
        eps=eps,
        depth=depth,
        enable_falling_constraint=True,
        use_heap=use_heap,
        rule_list_mode=rule_list_mode,
        branching_cost=branching_cost,
        min_support=min_support,
        use_current_leaf_prob=True,
        budget_override=None,
        dump_rset_jsonl=dump_path,
        max_cache_size=max_cache_size,
    )

In [4]:
result

{'best_loss': 4.9999999999999996e-06,
 'pmax': 1.0,
 'rset_size': 41,
 'best_tree_str': 'Node(feature=0)\n  Node(feature=3)\n    Node(feature=6)\n      Leaf(pred_prob=1.000, objective=0.000)\n      Node(feature=7)\n        Node(feature=8)\n          Leaf(pred_prob=0.000, objective=0.000)\n          Leaf(pred_prob=1.000, objective=0.000)\n        Node(feature=8)\n          Leaf(pred_prob=1.000, objective=0.000)\n          Leaf(pred_prob=0.000, objective=0.000)\n    Node(feature=4)\n      Node(feature=5)\n        Leaf(pred_prob=0.000, objective=0.000)\n        Leaf(pred_prob=1.000, objective=0.000)\n      Node(feature=5)\n        Leaf(pred_prob=1.000, objective=0.000)\n        Leaf(pred_prob=0.000, objective=0.000)\n  Node(feature=1)\n    Node(feature=2)\n      Leaf(pred_prob=0.000, objective=0.000)\n      Leaf(pred_prob=1.000, objective=0.000)\n    Node(feature=2)\n      Leaf(pred_prob=1.000, objective=0.000)\n      Leaf(pred_prob=0.000, objective=0.000)\n',
 'cache_size': 25505,
 'cach

In [5]:
print(result['best_tree_str'])

Node(feature=0)
  Node(feature=3)
    Node(feature=6)
      Leaf(pred_prob=1.000, objective=0.000)
      Node(feature=7)
        Node(feature=8)
          Leaf(pred_prob=0.000, objective=0.000)
          Leaf(pred_prob=1.000, objective=0.000)
        Node(feature=8)
          Leaf(pred_prob=1.000, objective=0.000)
          Leaf(pred_prob=0.000, objective=0.000)
    Node(feature=4)
      Node(feature=5)
        Leaf(pred_prob=0.000, objective=0.000)
        Leaf(pred_prob=1.000, objective=0.000)
      Node(feature=5)
        Leaf(pred_prob=1.000, objective=0.000)
        Leaf(pred_prob=0.000, objective=0.000)
  Node(feature=1)
    Node(feature=2)
      Leaf(pred_prob=0.000, objective=0.000)
      Leaf(pred_prob=1.000, objective=0.000)
    Node(feature=2)
      Leaf(pred_prob=1.000, objective=0.000)
      Leaf(pred_prob=0.000, objective=0.000)



In [6]:
lam=0.0
eps=0.01
depth=100
branching_cost=1.0
min_support=0.0
rule_list_mode=False
use_heap=True
max_cache_size=10**7
dump_path='/dev/null'

result = gravitree.run_from_xy(
        X.values,
        y.values,
        lam=lam,
        eps=eps,
        depth=depth,
        enable_falling_constraint=True,
        use_heap=use_heap,
        rule_list_mode=rule_list_mode,
        branching_cost=branching_cost,
        min_support=min_support,
        use_current_leaf_prob=True,
        budget_override=None,
        dump_rset_jsonl=dump_path,
        max_cache_size=max_cache_size,
    )

In [7]:
result

{'best_loss': 0.4375,
 'pmax': 0.5625,
 'rset_size': 2,
 'best_tree_str': 'Leaf(pred_prob=0.562, objective=0.438)\n',
 'cache_size': 39041,
 'cache_hits': 98701617,
 'cache_misses': 99540,
 'non_falling_new': 0}

In [8]:
print(result['best_tree_str'])

Leaf(pred_prob=0.562, objective=0.438)



In [9]:
X, y = generate_XOR_truth_table(spine_length=1, n_xor=2)
df = X.copy()
df['y'] = y
df

Features: ['X0', 'X1', 'X2']
Spine variables: [0]


,X0,X1,X2,y
0,0,0,0,1
1,0,0,1,1
2,0,1,0,1
3,0,1,1,1
4,1,0,0,0
5,1,0,1,1
6,1,1,0,1
7,1,1,1,0


In [10]:
lam=0.0
eps=0.01
depth=10
branching_cost=1e-6
min_support=0.
rule_list_mode=False
use_heap=True
max_cache_size=10**7
dump_path='/dev/null'

result = gravitree.run_from_xy(
        X.values,
        y.values,
        lam=lam,
        eps=eps,
        depth=depth,
        enable_falling_constraint=True,
        use_heap=use_heap,
        rule_list_mode=rule_list_mode,
        branching_cost=branching_cost,
        min_support=min_support,
        use_current_leaf_prob=True,
        budget_override=None,
        dump_rset_jsonl=dump_path,
        max_cache_size=max_cache_size,
    )
print(result['best_tree_str'])

Node(feature=0)
  Leaf(pred_prob=1.000, objective=0.000)
  Node(feature=1)
    Node(feature=2)
      Leaf(pred_prob=0.000, objective=0.000)
      Leaf(pred_prob=1.000, objective=0.000)
    Node(feature=2)
      Leaf(pred_prob=1.000, objective=0.000)
      Leaf(pred_prob=0.000, objective=0.000)



In [11]:
lam=0.0
eps=0.01
depth=100
branching_cost=1.0
min_support=0.0
rule_list_mode=False
use_heap=True
max_cache_size=10**7
dump_path='/home/users/zwb3/research/falling-models/experiments/rsets/and_truth_table_list.jsonl'

result = gravitree.run_from_xy(
        X.values,
        y.values,
        lam=lam,
        eps=eps,
        depth=depth,
        enable_falling_constraint=False,
        use_heap=use_heap,
        rule_list_mode=rule_list_mode,
        branching_cost=branching_cost,
        min_support=min_support,
        use_current_leaf_prob=True,
        budget_override=None,
        dump_rset_jsonl=dump_path,
        max_cache_size=max_cache_size,
    )

In [12]:
result

{'best_loss': 0.25,
 'pmax': 0.75,
 'rset_size': 2,
 'best_tree_str': 'Leaf(pred_prob=0.750, objective=0.250)\n',
 'cache_size': 45,
 'cache_hits': 283,
 'cache_misses': 98,
 'non_falling_new': 0}

In [13]:
print(result['best_tree_str'])

Leaf(pred_prob=0.750, objective=0.250)

